In [2]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 6.4 MB/s eta 0:00:00m eta 0:00:010:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 17.5 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008707 sha256=4f1e74bca74b3e98a93b22ff93bfcdbbf443a4ec1485cb6750c2c27e153845a2
  Stored in directory: /home/avinash/.cache/pip/wheels/f4/ca/ea/203f40b3e935bbf99bee851c2f4a87d22996ab8212d367ce58
Successfully built pyspark


## Working with RDD

In [1]:
from pyspark.sql import SparkSession

# Create Spark Context
spark = SparkSession.builder \
    .appName("LocalPySparkApp") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/19 23:59:05 WARN Utils: Your hostname, avinash, resolves to a loopback address: 127.0.1.1; using 192.168.29.70 instead (on interface wlp3s0)
26/03/19 23:59:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 23:59:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Read tripadvisor review file
review_rdd = sc.textFile("tripadvisor_review.txt")

In [3]:
type(review_rdd)

pyspark.core.rdd.RDD

In [4]:
# check first record
review_rdd.first()

'Nice place Better than some reviews give it credit for. Overall, the rooms were a bit small but nice. Everything was clean, the view was wonderful and it is very well located (the Prudential Center makes shopping and eating easy and the T is nearby for jaunts out and about the city). Overall, it was a good experience and the staff was quite friendly. '

In [5]:
# check inital 2 records
review_rdd.take(2)

['Nice place Better than some reviews give it credit for. Overall, the rooms were a bit small but nice. Everything was clean, the view was wonderful and it is very well located (the Prudential Center makes shopping and eating easy and the T is nearby for jaunts out and about the city). Overall, it was a good experience and the staff was quite friendly. ',
 'what a surprise What a surprise the Sheraton was after reading some of the reviews. it would appear there is a massive difference in the rooms, the South tower being the best. Check in was very efficient and the room was lovely, very large with the most comfortable beds ever. The hotel as stated is in a fantastic location and the Wrentham Village outlet is well worth a visit for bargain shopping ( the bus picks up outside). The hotel bar is a little pricey ( not helped by the current dollar rate) but is a nice place to relax after a busy day shopping. There is a number of restaurants close by. A cab from the airport to the hotel can

In [6]:
# Count reviews in the file
review_rdd.count()

332

In [7]:
# Tokenization
tokenized_reviews = review_rdd.flatMap(lambda review: review.lower().split(" "))  

print(tokenized_reviews.take(20))

['nice', 'place', 'better', 'than', 'some', 'reviews', 'give', 'it', 'credit', 'for.', 'overall,', 'the', 'rooms', 'were', 'a', 'bit', 'small', 'but', 'nice.', 'everything']


In [8]:
# Map Operation
mapped_reviews = tokenized_reviews.map(lambda x: (x,1)) 
mapped_reviews.take(5)

[('nice', 1), ('place', 1), ('better', 1), ('than', 1), ('some', 1)]

In [9]:
# Reduce Operation - using reduceByKey()
word_count = mapped_reviews.reduceByKey(lambda x,y: x+y) 

print(word_count.take(24))

[('place', 68), ('than', 85), ('give', 20), ('it', 457), ('were', 455), ('bit', 48), ('small', 81), ('but', 384), ('nice.', 19), ('everything', 33), ('view', 85), ('wonderful', 29), ('and', 1728), ('well', 61), ('center', 73), ('eating', 6), ('easy', 23), ('t', 58), ('for', 676), ('about', 110), ('good', 173), ('experience', 22), ('staff', 144), ('', 538)]


## Improve the Map-Reduce Code

In [10]:
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/avinash/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [11]:
stops = set(stopwords.words("english"))

print(stops)

{"shan't", 'very', "she'd", 'now', "mightn't", 'wasn', 'during', 'all', 'at', 'hasn', "we're", "weren't", 'wouldn', 'any', 'have', 'against', "needn't", 'just', 'same', 'further', 'for', 'not', 'won', 'so', "we've", "wouldn't", "i've", 'an', 'do', "should've", 'shan', 'as', 'does', 's', 'will', 'you', "they've", 'ain', 'me', 'below', 'other', 'this', 'if', "don't", 'while', 'down', 'had', 'off', 'why', 'those', 'because', 'about', 'by', 'haven', 'mightn', "haven't", "he'd", 'such', "wasn't", "hadn't", "you're", "he's", 'his', "i'm", 'm', 'am', 'she', "you'll", 'than', 'your', 'but', 'after', 'is', "shouldn't", "couldn't", 'has', 'with', 'only', 'under', "doesn't", 'over', 'itself', 'i', 'their', 'don', 'to', 'were', 'some', 'yourselves', "aren't", 'o', 'we', 'these', "they'll", 'until', 'nor', 'did', 'above', 'was', 'myself', 'should', 'hers', 'here', 'themselves', 'on', 'being', "she'll", 'each', 'whom', 'and', 'd', 'our', 'out', 'of', 'doesn', 're', 'a', "you've", 'needn', "hasn't", 

In [12]:
punc = list(string.punctuation)
print(punc)

['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '^', '_', '`', '{', '|', '}', '~']


In [13]:
# tokenization
tokens = review_rdd.flatMap(lambda x:wordpunct_tokenize(x.lower()))

In [14]:
# Filter and perform
stops = list(stops) + punc + [').',]

word_count = tokens.filter(
    lambda x: x not in stops
).map(
    lambda x: (x,1)
).reduceByKey(
    lambda x,y:x+y
)

print(word_count.take(10))

[('place', 79), ('give', 20), ('overall', 47), ('bit', 49), ('small', 115), ('everything', 38), ('clean', 105), ('view', 112), ('wonderful', 40), ('well', 97)]


## Create PySpark DataFram

In [15]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataFrameExample").getOrCreate()

data = [
    (1, "Avinash", 28, 50000),
    (2, "Ryan", 25, 45000),
    (3, "Alice", 30, 60000)
]

columns = ["id", "name", "age", "salary"]

# Create a PySpark DataFrame
df = spark.createDataFrame(data, columns)

df.show()

26/03/19 23:59:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+---+-------+---+------+
| id|   name|age|salary|
+---+-------+---+------+
|  1|Avinash| 28| 50000|
|  2|   Ryan| 25| 45000|
|  3|  Alice| 30| 60000|
+---+-------+---+------+



## Read CSV and Parquet File

In [16]:
# Read CSV file
df_csv = spark.read.csv("employee.csv", header=True, inferSchema=True)
df_csv.show()

+-------------+----+-------+------+----------+-----+-----------------+
|         name| age| income|gender|department|grade|performance_score|
+-------------+----+-------+------+----------+-----+-----------------+
|  Allen Smith|45.0|    NaN|   NaN|Operations|   G3|              723|
|      S Kumar| NaN|16000.0|     F|   Finance|   G0|              520|
|  Jack Morgan|32.0|35000.0|     M|   Finance|   G2|              674|
|    Ying Chin|45.0|65000.0|     F|     Sales|   G3|              556|
|Dheeraj Patel|30.0|42000.0|     F|Operations|   G2|              711|
|Satyam Sharma| NaN|62000.0|   NaN|     Sales|   G3|              649|
| James Authur|54.0|    NaN|     F|Operations|   G3|               53|
|   Josh Wills|54.0|52000.0|     F|   Finance|   G3|              901|
|     Leo Duck|23.0|98000.0|     M|     Sales|   G4|              709|
+-------------+----+-------+------+----------+-----+-----------------+



In [17]:
# Read parquet file
df_parquet = spark.read.parquet("employee.parquet")
df_parquet.show()

+-------------+----+-------+------+----------+-----+-----------------+
|         name| age| income|gender|department|grade|__index_level_0__|
+-------------+----+-------+------+----------+-----+-----------------+
|  Allen Smith|45.0|   NULL|  NULL|Operations|   G3|                0|
|      S Kumar|NULL|16000.0|     F|   Finance|   G0|                1|
|  Jack Morgan|32.0|35000.0|     M|   Finance|   G2|                2|
|    Ying Chin|45.0|65000.0|     F|     Sales|   G3|                3|
|Dheeraj Patel|30.0|42000.0|     F|Operations|   G2|                4|
|Satyam Sharma|NULL|62000.0|  NULL|     Sales|   G3|                5|
| James Authur|54.0|   NULL|     F|Operations|   G3|                6|
|   Josh Wills|54.0|52000.0|     F|   Finance|   G3|                7|
|     Leo Duck|23.0|98000.0|     M|     Sales|   G4|                8|
+-------------+----+-------+------+----------+-----+-----------------+



## Employee Data Analysis

In [18]:
# Create DataFrame
Employee = spark.read.csv("employee_data.csv", header=True, inferSchema=True)

In [19]:
Employee.printSchema()

root
 |-- Date_Of_Birth: date (nullable = true)
 |-- Employee_ID: integer (nullable = true)
 |-- Gender: integer (nullable = true)
 |-- Performance_Score: integer (nullable = true)
 |-- Salary: integer (nullable = true)
 |-- State: string (nullable = true)



In [20]:
# Count number of records in  dataframe
Employee.count()

1000

In [21]:
type(Employee)

pyspark.sql.classic.dataframe.DataFrame

In [22]:
# Show top 20 records
Employee.show()

+-------------+-----------+------+-----------------+------+--------------+
|Date_Of_Birth|Employee_ID|Gender|Performance_Score|Salary|         State|
+-------------+-----------+------+-----------------+------+--------------+
|   1973-08-26|     358043|     0|               63|  9606|       Arizona|
|   2005-12-03|     641889|     0|               81|  4757|North Carolina|
|   2002-12-17|      72630|     1|                2|  9800|       Florida|
|   1973-08-14|      27981|     0|               21|  9846| West Virginia|
|   1960-11-08|     833993|     1|               86|  7783|      Maryland|
|   2002-09-13|     196765|     1|               31|  5880|     Louisiana|
|   1998-02-19|     435054|     1|               50|  9160|       Vermont|
|   1964-10-30|     631721|     0|               48|  1235|      Virginia|
|   1994-06-27|     685862|     1|                4|  7054|      Michigan|
|   1984-01-10|     115483|     0|               69|  2258|      Kentucky|
|   1997-10-22|      3767

In [25]:
# Describe the dataframe
Employee.describe().show()

26/03/19 23:59:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+------------------+-----------------+-----------------+-------+
|summary|      Employee_ID|            Gender|Performance_Score|           Salary|  State|
+-------+-----------------+------------------+-----------------+-----------------+-------+
|  count|             1000|              1000|             1000|             1000|   1000|
|   mean|       486216.208|             0.479|           49.925|         4942.091|   NULL|
| stddev|284140.0128563182|0.4998087722407528|28.50340012733252|2843.955859300324|   NULL|
|    min|             1020|                 0|                0|               26|Alabama|
|    max|           999774|                 1|               99|             9990|Wyoming|
+-------+-----------------+------------------+-----------------+-----------------+-------+



In [33]:
Employee.columns

['Date_Of_Birth',
 'Employee_ID',
 'Gender',
 'Performance_Score',
 'Salary',
 'State']

## select() Operation

In [35]:
# Select Employee ID, Salary, and Bonus
Employee.select('Employee_ID','Salary').show()

+-----------+------+
|Employee_ID|Salary|
+-----------+------+
|     358043|  9606|
|     641889|  4757|
|      72630|  9800|
|      27981|  9846|
|     833993|  7783|
|     196765|  5880|
|     435054|  9160|
|     631721|  1235|
|     685862|  7054|
|     115483|  2258|
|      37670|  2706|
|     120969|  6668|
|     719834|  4541|
|     267266|  4508|
|     984138|  7669|
|     740166|  4938|
|      33625|  7355|
|     192625|  1334|
|     275997|  5662|
|     214813|  2840|
+-----------+------+
only showing top 20 rows


## PySpark DataFrame, RDD, and Pandas DataFrame

In [36]:
# PySpark DataFrame to RDD
emp_rdd = Employee.rdd

type(emp_rdd)

pyspark.core.rdd.RDD

In [37]:
# RDD to PySpark DataFrame
emp_df = emp_rdd.toDF()
type(emp_df)

pyspark.sql.classic.dataframe.DataFrame

In [38]:
# PySpark DataFrame to Pandas DataFrame
emp_pd = Employee.toPandas()
type(emp_pd)

pandas.core.frame.DataFrame

In [39]:
# Pandas DataFrame to PySpark DataFrame to 
emp_pd_df = spark.createDataFrame(emp_pd)
type(emp_pd_df)

pyspark.sql.classic.dataframe.DataFrame

## filter() Operation

In [40]:
# List of emplyees who are male and their salary is greater than 9000
Employee.filter((Employee.Gender == 1) & (Employee.Salary>9000)).show()

+-------------+-----------+------+-----------------+------+------------+
|Date_Of_Birth|Employee_ID|Gender|Performance_Score|Salary|       State|
+-------------+-----------+------+-----------------+------+------------+
|   2002-12-17|      72630|     1|                2|  9800|     Florida|
|   1998-02-19|     435054|     1|               50|  9160|     Vermont|
|   1963-04-03|     907384|     1|               84|  9208|      Oregon|
|   1992-10-23|     437985|     1|               65|  9481|North Dakota|
|   1966-07-16|     496107|     1|               64|  9080|      Nevada|
|   1994-01-22|     643290|     1|               87|  9572|      Hawaii|
|   1990-01-06|     545905|     1|               47|  9500|      Alaska|
|   2000-12-15|     735664|     1|               17|  9319|       Idaho|
|   2003-04-08|     244052|     1|               87|  9517|    Nebraska|
|   1967-06-11|      71380|     1|               13|  9990|    New York|
|   2000-03-31|      41579|     1|               41

In [41]:
# List of emplyees who are female and their performance score is greater than 90
Employee.filter((Employee.Gender == 0) & (Employee.Performance_Score > 90)).show()

+-------------+-----------+------+-----------------+------+--------------+
|Date_Of_Birth|Employee_ID|Gender|Performance_Score|Salary|         State|
+-------------+-----------+------+-----------------+------+--------------+
|   1978-05-06|     478364|     0|               99|  1801|    California|
|   1984-10-20|     618488|     0|               93|  8803|      Maryland|
|   1979-06-14|     983405|     0|               98|  2968| West Virginia|
|   1972-03-07|     122704|     0|               95|  3481|     Wisconsin|
|   1999-02-04|     844427|     0|               91|  9246|         Texas|
|   1988-09-11|     121070|     0|               95|  9248|         Idaho|
|   1997-12-19|     107576|     0|               97|  9656|       Montana|
|   1982-12-14|     446953|     0|               99|  4773|   Connecticut|
|   1998-05-14|     629533|     0|               92|  8736|      Oklahoma|
|   1981-11-29|      80049|     0|               99|  8943|       Arizona|
|   1982-11-16|     20939

## Create new column using withColumn()

In [42]:
from pyspark.sql.functions import datediff, current_timestamp, year
from pyspark.sql.functions import floor, round

In [43]:
# Create column using withColumn()
Employee = Employee.withColumn("Age", floor(datediff(current_timestamp(),Employee.Date_Of_Birth)/365))

In [44]:
Employee.show(10)

+-------------+-----------+------+-----------------+------+--------------+---+
|Date_Of_Birth|Employee_ID|Gender|Performance_Score|Salary|         State|Age|
+-------------+-----------+------+-----------------+------+--------------+---+
|   1973-08-26|     358043|     0|               63|  9606|       Arizona| 52|
|   2005-12-03|     641889|     0|               81|  4757|North Carolina| 20|
|   2002-12-17|      72630|     1|                2|  9800|       Florida| 23|
|   1973-08-14|      27981|     0|               21|  9846| West Virginia| 52|
|   1960-11-08|     833993|     1|               86|  7783|      Maryland| 65|
|   2002-09-13|     196765|     1|               31|  5880|     Louisiana| 23|
|   1998-02-19|     435054|     1|               50|  9160|       Vermont| 28|
|   1964-10-30|     631721|     0|               48|  1235|      Virginia| 61|
|   1994-06-27|     685862|     1|                4|  7054|      Michigan| 31|
|   1984-01-10|     115483|     0|               69|

In [45]:
# Create column using withColumn()
Employee = Employee.withColumn("Bonus", round(Employee.Salary*0.25))

In [46]:
# Create column using withColumn()
Employee = Employee.withColumn("birth_year", year(Employee.Date_Of_Birth))

In [47]:
Employee.show(10)

+-------------+-----------+------+-----------------+------+--------------+---+------+----------+
|Date_Of_Birth|Employee_ID|Gender|Performance_Score|Salary|         State|Age| Bonus|birth_year|
+-------------+-----------+------+-----------------+------+--------------+---+------+----------+
|   1973-08-26|     358043|     0|               63|  9606|       Arizona| 52|2402.0|      1973|
|   2005-12-03|     641889|     0|               81|  4757|North Carolina| 20|1189.0|      2005|
|   2002-12-17|      72630|     1|                2|  9800|       Florida| 23|2450.0|      2002|
|   1973-08-14|      27981|     0|               21|  9846| West Virginia| 52|2462.0|      1973|
|   1960-11-08|     833993|     1|               86|  7783|      Maryland| 65|1946.0|      1960|
|   2002-09-13|     196765|     1|               31|  5880|     Louisiana| 23|1470.0|      2002|
|   1998-02-19|     435054|     1|               50|  9160|       Vermont| 28|2290.0|      1998|
|   1964-10-30|     631721|   

## User Defined Functions

In [48]:
# Error
Employee["Employee_Class"] = Employee.select("Salary").apply(lambda x:'A' if x>=5000  else 'B' if (x<5000 and x>=2000) else 'C')

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `apply` is not supported.

In [49]:
from pyspark.sql.functions import udf

# User defined function using udf
salary_grade_udf = udf(lambda x:'A' if x>=5000  else 'B' if x<5000 and x>=2000 else 'C')

# Create column using withColumn()
Employee=Employee.withColumn("Grade", salary_grade_udf(Employee.Salary))

Employee.show(5)

+-------------+-----------+------+-----------------+------+--------------+---+------+----------+-----+
|Date_Of_Birth|Employee_ID|Gender|Performance_Score|Salary|         State|Age| Bonus|birth_year|Grade|
+-------------+-----------+------+-----------------+------+--------------+---+------+----------+-----+
|   1973-08-26|     358043|     0|               63|  9606|       Arizona| 52|2402.0|      1973|    A|
|   2005-12-03|     641889|     0|               81|  4757|North Carolina| 20|1189.0|      2005|    B|
|   2002-12-17|      72630|     1|                2|  9800|       Florida| 23|2450.0|      2002|    A|
|   1973-08-14|      27981|     0|               21|  9846| West Virginia| 52|2462.0|      1973|    A|
|   1960-11-08|     833993|     1|               86|  7783|      Maryland| 65|1946.0|      1960|    A|
+-------------+-----------+------+-----------------+------+--------------+---+------+----------+-----+
only showing top 5 rows


## grouby() and orderby() operations

In [50]:
# Group By operations
Employee.groupBy(Employee.State).agg({"Salary": "avg"}).orderBy("avg(Salary)",ascending=True).show()

+--------------+------------------+
|         State|       avg(Salary)|
+--------------+------------------+
|      Oklahoma|          3793.625|
| Massachusetts|           3842.72|
|  Pennsylvania| 4041.285714285714|
|      Illinois|4143.9473684210525|
|South Carolina| 4167.777777777777|
|     Tennessee|         4202.9375|
|   Connecticut|           4210.92|
|    Washington| 4276.571428571428|
|       Arizona| 4308.083333333333|
|    New Jersey| 4374.307692307692|
|      Virginia| 4414.307692307692|
|  South Dakota| 4508.333333333333|
|          Ohio|            4514.2|
|  Rhode Island| 4540.333333333333|
|    New Mexico| 4562.466666666666|
| New Hampshire| 4570.785714285715|
|         Idaho| 4593.470588235294|
|          Iowa| 4597.821428571428|
|          Utah|            4651.0|
|        Nevada| 4655.133333333333|
+--------------+------------------+
only showing top 20 rows


## Join Operations

In [51]:
# Load another employee data csv file
emp_data = spark.read.csv("emp_dpt_data.csv", header=True, inferSchema=True)

emp_data.show()

+------+-------+-------+
|emp_id|   name|dept_id|
+------+-------+-------+
|     1|Avinash|    101|
|     2|   Riya|    102|
|     3|  Karan|    103|
|     4|   Neha|    107|
+------+-------+-------+



In [52]:
# Load department data csv file
dept_data = spark.read.csv("department_data.csv", header=True, inferSchema=True)

dept_data.show()

+-------+---------------+
|dept_id|department_name|
+-------+---------------+
|    101|             HR|
|    102|             IT|
|    103|        Finance|
|    104|      Marketing|
|    105|     Operations|
|    106|      Analytics|
+-------+---------------+



In [53]:
emp_data.join(dept_data, on="dept_id", how="inner").show()

+-------+------+-------+---------------+
|dept_id|emp_id|   name|department_name|
+-------+------+-------+---------------+
|    101|     1|Avinash|             HR|
|    102|     2|   Riya|             IT|
|    103|     3|  Karan|        Finance|
+-------+------+-------+---------------+



In [54]:
emp_data.join(dept_data, on="dept_id", how="left").show()

+-------+------+-------+---------------+
|dept_id|emp_id|   name|department_name|
+-------+------+-------+---------------+
|    101|     1|Avinash|             HR|
|    102|     2|   Riya|             IT|
|    103|     3|  Karan|        Finance|
|    107|     4|   Neha|           NULL|
+-------+------+-------+---------------+



In [55]:
emp_data.join(dept_data, on="dept_id", how="right").show()

+-------+------+-------+---------------+
|dept_id|emp_id|   name|department_name|
+-------+------+-------+---------------+
|    101|     1|Avinash|             HR|
|    102|     2|   Riya|             IT|
|    103|     3|  Karan|        Finance|
|    104|  NULL|   NULL|      Marketing|
|    105|  NULL|   NULL|     Operations|
|    106|  NULL|   NULL|      Analytics|
+-------+------+-------+---------------+



## dropna(), fillna(), and replace() operations

In [87]:
data = [
    (1, "Ryan", "M", 28, None),
    (2, "Samira", "F", None, 45000),
    (3, None, "M", 30, 60000)
]

columns = ["id", "name", "gender", "age", "salary"]

df_null = spark.createDataFrame(data, columns)
df_null.show()

+---+------+------+----+------+
| id|  name|gender| age|salary|
+---+------+------+----+------+
|  1|  Ryan|     M|  28|  NULL|
|  2|Samira|     F|NULL| 45000|
|  3|  NULL|     M|  30| 60000|
+---+------+------+----+------+



In [88]:
df_null.fillna({"name": "Unknown", "age": 0, "salary": 0}).show()

+---+-------+------+---+------+
| id|   name|gender|age|salary|
+---+-------+------+---+------+
|  1|   Ryan|     M| 28|     0|
|  2| Samira|     F|  0| 45000|
|  3|Unknown|     M| 30| 60000|
+---+-------+------+---+------+



In [89]:
df_null.dropna().show()

+---+----+------+---+------+
| id|name|gender|age|salary|
+---+----+------+---+------+
+---+----+------+---+------+



In [95]:
df_null.replace({"M": "Male", "F": "Female"}, subset=["gender"]).show()

+---+------+------+----+------+
| id|  name|gender| age|salary|
+---+------+------+----+------+
|  1|  Ryan|  Male|  28|  NULL|
|  2|Samira|Female|NULL| 45000|
|  3|  NULL|  Male|  30| 60000|
+---+------+------+----+------+



## Broadcast Variables and Accumulators

In [61]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("BroadcastConfigExample").getOrCreate()
sc = spark.sparkContext

# configuration data
config = {
    "tax_rate": 0.05,
    "currency": "INR"
}

# Create Broadcast variable
broadcast_config = sc.broadcast(config)

data = [
    (1, "Amit", 50000),
    (2, "Ryan", 60000),
    (3, "Alice", 45000)
]

rdd = sc.parallelize(data)

# Use broadcasted config inside transformation
result = rdd.map(
    lambda x: (
        x[0],
        x[1],
        x[2],
        x[2] * broadcast_config.value["tax_rate"],
        broadcast_config.value["currency"]
    )
).collect()

# Show the results
for row in result:
    print(row)

26/03/20 00:01:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


(1, 'Amit', 50000, 2500.0, 'INR')
(2, 'Ryan', 60000, 3000.0, 'INR')
(3, 'Alice', 45000, 2250.0, 'INR')


In [62]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("AccumulatorExample").getOrCreate()
sc = spark.sparkContext

# Create RDD
data = [10, 20, 30, 40, 50]
rdd = sc.parallelize(data)

# Create accumulator
sum_acc = sc.accumulator(0)

# Add values to accumulator
def add_to_acc(x):
    global sum_acc
    sum_acc += x

rdd.foreach(add_to_acc)

print("Sum of all values:", sum_acc.value)

26/03/20 00:01:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Sum of all values: 150


## Scalable Machine Learning Models

### Classification Problem - Churn prediction

In [63]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [64]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Churn Classification") \
    .master("local[*]") \
    .getOrCreate()

26/03/20 00:01:39 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [65]:
data = spark.read.csv("Churn_Modelling.csv", header=True, inferSchema=True)

In [66]:
data = data.select(
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "Exited"
)

data.show(5)

+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        619|   France|Female| 42|     2|      0.0|            1|        1|             1|      101348.88|     1|
|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|
|        699|   France|Female| 39|     1|      0.0|            2|        0|             0|       93826.63|     0|
|        850|    Spain|Female| 43|     2|125510.82|            1|        1|             1|        79084.1|     0|
+-----------+---------+------+---+------+---------+-------------+---------+-------------

In [67]:
data.printSchema()

root
 |-- CreditScore: integer (nullable = true)
 |-- Geography: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Tenure: integer (nullable = true)
 |-- Balance: double (nullable = true)
 |-- NumOfProducts: integer (nullable = true)
 |-- HasCrCard: integer (nullable = true)
 |-- IsActiveMember: integer (nullable = true)
 |-- EstimatedSalary: double (nullable = true)
 |-- Exited: integer (nullable = true)



In [68]:
# Convert categorical columns into numeric
geo_indexer = StringIndexer(inputCol="Geography", outputCol="GeographyIndex")
data = geo_indexer.fit(data).transform(data)

gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex")
data = gender_indexer.fit(data).transform(data)

In [69]:
# Create feature vector
feature_columns = [
    "CreditScore",
    "GeographyIndex",
    "GenderIndex",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
data = assembler.transform(data)

In [70]:
# Final dataset for model
final_data = data.select("features", col("Exited").alias("label"))
final_data.show(5, truncate=False)

+-------------------------------------------------------+-----+
|features                                               |label|
+-------------------------------------------------------+-----+
|[619.0,0.0,1.0,42.0,2.0,0.0,1.0,1.0,1.0,101348.88]     |1    |
|[608.0,2.0,1.0,41.0,1.0,83807.86,1.0,0.0,1.0,112542.58]|0    |
|[502.0,0.0,1.0,42.0,8.0,159660.8,3.0,1.0,0.0,113931.57]|1    |
|[699.0,0.0,1.0,39.0,1.0,0.0,2.0,0.0,0.0,93826.63]      |0    |
|[850.0,2.0,1.0,43.0,2.0,125510.82,1.0,1.0,1.0,79084.1] |0    |
+-------------------------------------------------------+-----+
only showing top 5 rows


In [71]:
# Split into train and test
train_data, test_data = final_data.randomSplit([0.7, 0.3], seed=123)

In [72]:
# Create decision tree model and make prediction
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features")
dt_model = dt.fit(train_data)

dt_predictions = dt_model.transform(test_data)
dt_predictions.select("label", "prediction", "probability").show(10, truncate=False)

+-----+----------+----------------------------------------+
|label|prediction|probability                             |
+-----+----------+----------------------------------------+
|0    |0.0       |[0.9799283154121864,0.02007168458781362]|
|0    |0.0       |[0.9799283154121864,0.02007168458781362]|
|0    |0.0       |[0.9799283154121864,0.02007168458781362]|
|0    |0.0       |[0.9799283154121864,0.02007168458781362]|
|1    |0.0       |[0.8368026644462948,0.16319733555370525]|
|0    |0.0       |[0.8368026644462948,0.16319733555370525]|
|1    |0.0       |[0.5132743362831859,0.48672566371681414]|
|1    |1.0       |[0.13448275862068965,0.8655172413793103]|
|0    |0.0       |[0.9799283154121864,0.02007168458781362]|
|1    |1.0       |[0.13448275862068965,0.8655172413793103]|
+-----+----------+----------------------------------------+
only showing top 10 rows


In [73]:
# Evaluate decition tree model
evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

dt_accuracy = evaluator_accuracy.evaluate(dt_predictions)
print("Decision Tree Accuracy:", dt_accuracy)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

dt_precision = evaluator_precision.evaluate(dt_predictions)
print("Decision Tree Precision:", dt_precision)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

dt_recall = evaluator_recall.evaluate(dt_predictions)
print("Decision Tree Recall:", dt_recall)

Decision Tree Accuracy: 0.8515981735159818
Decision Tree Precision: 0.8449380854128097
Decision Tree Recall: 0.8515981735159818


In [74]:
# Create logistic reegression model and make prediction
lr = LogisticRegression(labelCol="label", featuresCol="features")
lr_model = lr.fit(train_data)

lr_predictions = lr_model.transform(test_data)
# lr_predictions.select("label", "prediction", "probability").show(10, truncate=False)

In [75]:
evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

evaluator_recall = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

lr_accuracy = evaluator_accuracy.evaluate(lr_predictions)
lr_precision = evaluator_precision.evaluate(lr_predictions)
lr_recall = evaluator_recall.evaluate(lr_predictions)

print("Logistic Regression Accuracy :", lr_accuracy)
print("Logistic Regression Precision:", lr_precision)
print("Logistic Regression Recall   :", lr_recall)

Logistic Regression Accuracy : 0.7997390737116764
Logistic Regression Precision: 0.7635063109792452
Logistic Regression Recall   : 0.7997390737116765


### Regression Problem - Predicting Medical Insurance Charges

In [76]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [77]:
# Create Spark Session
spark = SparkSession.builder \
    .appName("Medical Insurance Regression") \
    .master("local[*]") \
    .getOrCreate()

26/03/20 00:01:53 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [78]:
data = spark.read.csv("insurance.csv", header=True, inferSchema=True)
data.show(5)

+---+------+------+--------+------+---------+-----------+
|age|   sex|   bmi|children|smoker|   region|    charges|
+---+------+------+--------+------+---------+-----------+
| 19|female|  27.9|       0|   yes|southwest|  16884.924|
| 18|  male| 33.77|       1|    no|southeast|  1725.5523|
| 28|  male|  33.0|       3|    no|southeast|   4449.462|
| 33|  male|22.705|       0|    no|northwest|21984.47061|
| 32|  male| 28.88|       0|    no|northwest|  3866.8552|
+---+------+------+--------+------+---------+-----------+
only showing top 5 rows


In [79]:
print(data.columns)
data.printSchema()

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']
root
 |-- age: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- children: integer (nullable = true)
 |-- smoker: string (nullable = true)
 |-- region: string (nullable = true)
 |-- charges: double (nullable = true)



In [80]:
# Convert categorical columns into numeric
sex_indexer = StringIndexer(inputCol="sex", outputCol="sex_index")
data = sex_indexer.fit(data).transform(data)


smoker_indexer = StringIndexer(inputCol="smoker", outputCol="smoker_index")
data = smoker_indexer.fit(data).transform(data)


region_indexer = StringIndexer(inputCol="region", outputCol="region_index")
data = region_indexer.fit(data).transform(data)

In [81]:
feature_cols = [
    "age",
    "sex_index",
    "bmi",
    "children",
    "smoker_index",
    "region_index"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(data)

final_data = data.select(
    "features",
    col("charges").alias("label")
)

final_data.show(5, truncate=False)

+-----------------------------+-----------+
|features                     |label      |
+-----------------------------+-----------+
|[19.0,1.0,27.9,0.0,1.0,2.0]  |16884.924  |
|[18.0,0.0,33.77,1.0,0.0,0.0] |1725.5523  |
|[28.0,0.0,33.0,3.0,0.0,0.0]  |4449.462   |
|[33.0,0.0,22.705,0.0,0.0,1.0]|21984.47061|
|[32.0,0.0,28.88,0.0,0.0,1.0] |3866.8552  |
+-----------------------------+-----------+
only showing top 5 rows


In [82]:
# Split into train and test
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

In [83]:
# Train Linear Regression model
lr = LinearRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train_data)

# Predict using Linear Regression
lr_predictions = lr_model.transform(test_data)
lr_predictions.select("label", "prediction").show(10, truncate=False)

26/03/20 00:01:55 WARN Instrumentation: [a4eadc1a] regParam is zero, which might cause numerical instability and overfitting.


+-----------+------------------+
|label      |prediction        |
+-----------+------------------+
|1135.9407  |2852.7563797479634|
|1141.4451  |4190.269041281295 |
|1149.3959  |6122.231774607215 |
|1532.4697  |4357.459227973843 |
|1837.2819  |7020.100910545414 |
|2322.6218  |5589.706529295378 |
|3704.3545  |5620.6700465366375|
|5699.8375  |8766.304620441453 |
|27346.04207|11069.80306464128 |
|9504.3103  |14487.890977448678|
+-----------+------------------+
only showing top 10 rows


In [84]:
# Evaluate Linear Regression - RMSE, R2, MAE
rmse_eval = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

lr_rmse = rmse_eval.evaluate(lr_predictions)
print("Linear Regression RMSE:", lr_rmse)

r2_eval = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="r2"
)

lr_r2 = r2_eval.evaluate(lr_predictions)
print("Linear Regression R2:", lr_r2)

mae_eval = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="mae"
)

lr_mae = mae_eval.evaluate(lr_predictions)
print("Linear Regression MAE:", lr_mae)

Linear Regression RMSE: 6240.095537633676
Linear Regression R2: 0.7190905506845703
Linear Regression MAE: 4279.1035566064


In [85]:
# Train Decision Tree model
dt = DecisionTreeRegressor(featuresCol="features", labelCol="label")
dt_model = dt.fit(train_data)

# Predict using Decision Tree
dt_predictions = dt_model.transform(test_data)

# Evaluate Decision Tree - RMSE, R2, MAE
dt_rmse = rmse_eval.evaluate(dt_predictions)
print("Decision Tree RMSE:", dt_rmse)

dt_r2 = r2_eval.evaluate(dt_predictions)
print("Decision Tree R2:", dt_r2)

dt_mae = mae_eval.evaluate(dt_predictions)
print("Decision Tree MAE:", dt_mae)

Decision Tree RMSE: 4880.190323919041
Decision Tree R2: 0.828186458590104
Decision Tree MAE: 2789.8505889573557
